In [0]:
from sklearn.ensemble import RandomForestRegressor
import mlflow.sklearn

df = spark.table("instagram.goldlayer.vw_health_behavior_analysis").toPandas()

# Features include your total_time_consumed calculation
X = df[['total_time_consumed', 'sleep_hours_per_night', 'exercise_hours_per_week', 'daily_steps_count']]
y = df['user_engagement_score']

with mlflow.start_run(run_name="Engagement_Regression"):
    model = RandomForestRegressor(n_estimators=100).fit(X, y)
    mlflow.sklearn.log_model(model, "model")
    
    # Save results
    predictions = df[['user_id']].copy()
    predictions['predicted_engagement_score'] = model.predict(X)
    spark.createDataFrame(predictions).write.mode("overwrite").saveAsTable("instagram.model_output.ml_engagement")

print("✅ Engagement Forecast Ready, bro!")